<div style="text-align:center">

# Proyecto 1  
## Clasificación de Riesgo Crediticio  

**Machine Learning**  
Universidad del Rosario

Vanessa Ochoa, Nilson Amaya, Juan Zamora

</div>

---

## Tabla de contenido

- [1. Introducción](#1-introducción)
- [2. Importación del dataset](#2-importación-del-dataset)
- [3. Análisis Exploratorio de Datos (EDA)](#3-eda)
- [4. Implementación de Modelos](#4-modelos)
- [5. Optimización de Hiperparámetros y Validación](#5-optimización)
- [6. Métricas de Evaluación y Decisión](#6-métricas)
- [7. Predicción en el conjunto de test](#7-test)


<a id="1-introducción"></a>
## 1. Introducción

El riesgo crediticio es uno de los principales problemas que enfrentan las instituciones financieras, ya que implica la posibilidad de que una persona no cumpla con sus obligaciones de pago. Poder anticipar este tipo de situaciones es clave para reducir pérdidas y mejorar la toma de decisiones.

En este proyecto se utiliza el dataset **"Give Me Some Credit"** con el objetivo de predecir si una persona tendrá dificultades financieras en los próximos dos años. Este problema se aborda como una tarea de **clasificación binaria**, donde el modelo debe identificar si un cliente presenta riesgo de incumplimiento o no.

Para resolver este problema se desarrolla un flujo completo de **Machine Learning**, que incluye:

- Análisis exploratorio de datos (EDA), tratamiento de valores faltantes y outliers.
- Implementación y comparación de modelos de clasificación: **k-NN** y **Regresión Logística**.
- Evaluación del impacto del tipo de escalamiento (**Normalización** vs. **Estandarización**) sobre el desempeño.
- Optimización de hiperparámetros mediante **Grid Search** y **validación cruzada estratificada**.
- Evaluación final con **métricas de clasificación** adecuadas para datos desbalanceados.

Finalmente, se comparan los modelos para identificar cuál ofrece el mejor desempeño en la predicción del riesgo crediticio.

<a id="2-importación-del-dataset"></a>
## 2. Importación del dataset

Antes de cargar el dataset, se importan las librerías necesarias para el desarrollo del proyecto.

In [ ]:
########################################################
## LIBRERÍAS                                          ##
########################################################

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Modelado y evaluación
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

A continuación cargamos los archivos `cs-training.csv` y `cs-test.csv` directamente desde Google Colab. La primera columna de ambos archivos es un índice sin nombre que se descarta al momento de la carga.

In [ ]:
########################################################
## CARGA DEL DATASET                                  ##
########################################################

from google.colab import files

# Subir archivos manualmente desde tu computador
uploaded = files.upload()  # seleccionar cs-training.csv y cs-test.csv

# Cargar los datasets
train = pd.read_csv("cs-training.csv", index_col=0)
test  = pd.read_csv("cs-test.csv",     index_col=0)

# Vista general del conjunto de entrenamiento
print("Dimensiones del conjunto de entrenamiento:", train.shape)
print("Dimensiones del conjunto de test:         ", test.shape)
train.info()

Revisamos un resumen estadístico de las variables para tener una primera idea de sus rangos, valores extremos y posibles inconsistencias.

In [ ]:
train.describe()

### 2.1 Descripción de las variables

- **SeriousDlqin2yrs:** Variable objetivo. Indica si la persona tuvo dificultades financieras graves (atraso ≥ 90 días) en los siguientes dos años. Toma valor 1 (sí incumplió) o 0 (no incumplió).

- **RevolvingUtilizationOfUnsecuredLines:** Proporción del saldo utilizado en tarjetas de crédito y líneas personales respecto al límite total disponible. En condiciones normales se expresa entre 0 y 1.

- **age:** Edad del titular del crédito en años.

- **NumberOfTime30-59DaysPastDueNotWorse:** Número de veces que el prestatario tuvo atrasos entre 30 y 59 días en los últimos dos años.

- **DebtRatio:** Relación entre las obligaciones financieras mensuales y el ingreso bruto mensual.

- **MonthlyIncome:** Ingreso mensual reportado por el prestatario.

- **NumberOfOpenCreditLinesAndLoans:** Número total de préstamos y líneas de crédito abiertas.

- **NumberOfTimes90DaysLate:** Número de veces que el prestatario tuvo atrasos de 90 días o más.

- **NumberRealEstateLoansOrLines:** Número de préstamos hipotecarios o líneas de crédito sobre vivienda.

- **NumberOfTime60-89DaysPastDueNotWorse:** Número de veces que el prestatario tuvo atrasos entre 60 y 89 días en los últimos dos años.

- **NumberOfDependents:** Número de personas que dependen económicamente del prestatario.

<a id="3-eda"></a>
## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Identificación y tratamiento de datos faltantes e imputación lógica

Antes de entrenar cualquier modelo es necesario identificar si existen variables con datos ausentes, ya que la mayoría de los algoritmos de Machine Learning no pueden trabajar con valores faltantes directamente.

A continuación identificamos el porcentaje de datos faltantes por variable:

In [ ]:
########################################################
## IDENTIFICACIÓN DE VALORES FALTANTES               ##
########################################################

faltantes  = train.isnull().sum()
porcentaje = (faltantes / len(train)) * 100

tabla_faltantes = pd.DataFrame({
    'Valores faltantes': faltantes,
    'Porcentaje (%)':    porcentaje.round(2)
})

print(tabla_faltantes[tabla_faltantes['Valores faltantes'] > 0])

Se identificaron dos variables con datos ausentes:

- **MonthlyIncome**: 19.82% de valores faltantes. Al analizar estos registros, se observa que las personas con ingreso desconocido tienen una tasa de incumplimiento ligeramente mayor que las personas con ingreso conocido. Esto indica que **los valores faltantes no son completamente aleatorios** y contienen información. Por esta razón, antes de imputar se crea una variable indicadora (`flag_ingreso_nulo`) que registra si el ingreso era originalmente desconocido.

- **NumberOfDependents**: 2.62% de valores faltantes. El porcentaje es bajo y no presenta un patrón diferenciado respecto al incumplimiento, por lo que se imputa directamente con la mediana.

En ambos casos se usa la **mediana** como valor de imputación porque, a diferencia de la media, es resistente a los valores extremos que caracterizan a las variables financieras.

In [ ]:
########################################################
## IMPUTACIÓN CON MEDIANA                            ##
########################################################

# Crear variable indicadora ANTES de imputar MonthlyIncome
# Registra si el ingreso era desconocido originalmente
train['flag_ingreso_nulo'] = train['MonthlyIncome'].isnull().astype(int)
test['flag_ingreso_nulo']  = test['MonthlyIncome'].isnull().astype(int)

# Calcular medianas sobre el conjunto de entrenamiento
median_monthly_income = train['MonthlyIncome'].median()
median_dependents     = train['NumberOfDependents'].median()

# Imputar en train
train['MonthlyIncome']       = train['MonthlyIncome'].fillna(median_monthly_income)
train['NumberOfDependents']  = train['NumberOfDependents'].fillna(median_dependents)

# Imputar en test con los mismos valores calculados en train
test['MonthlyIncome']        = test['MonthlyIncome'].fillna(median_monthly_income)
test['NumberOfDependents']   = test['NumberOfDependents'].fillna(median_dependents)

# Verificar
print("Faltantes en train después de imputación:", train.isnull().sum().sum())
print("Faltantes en test después de imputación: ", test.isnull().sum().sum())
print(f"\nMediana MonthlyIncome usada:      {median_monthly_income:.2f}")
print(f"Mediana NumberOfDependents usada: {median_dependents:.2f}")

### 3.2 Análisis de distribuciones y detección de outliers

Analizamos la forma de las distribuciones con boxplots para identificar valores extremos. A continuación se visualizan las 10 variables numéricas del dataset:

In [ ]:
########################################################
## BOXPLOTS — DETECCIÓN DE OUTLIERS                  ##
########################################################

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

columnas = [
    'RevolvingUtilizationOfUnsecuredLines', 'age',
    'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio',
    'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate', 'NumberOfDependents',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

for i, col in enumerate(columnas):
    axes[i].boxplot(train[col], patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red'))
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Valor')
    axes[i].grid(alpha=0.3)

fig.suptitle('Detección de Outliers por Variable', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

Los boxplots revelan la presencia de valores extremos en varias variables. El tratamiento de cada una se define considerando su **significado financiero**, no simplemente su magnitud:

- **RevolvingUtilizationOfUnsecuredLines**: en condiciones normales este ratio está entre 0 y 1 (0% a 100% del límite usado). Sin embargo, los valores superiores a 1 corresponden a personas que **superaron su límite de crédito**, lo cual es en sí mismo una señal de riesgo. Por eso no se restringe a 1 — hacerlo eliminaría esa señal. Se aplica un capado en 5 para descartar únicamente los valores claramente erróneos (máximo observado: 50,708).

- **DebtRatio**: el valor extremo de esta variable se explica porque el ratio se calcula como deuda dividida entre ingreso mensual. Cuando el ingreso es cero o nulo, el resultado es matemáticamente inestable y no representa una deuda real. Se aplica un capado en el percentil 99 para conservar la variabilidad real.

- **age**: se limita al rango 18–95 para eliminar edades imposibles o poco representativas.

- **MonthlyIncome**: se capea en el percentil 99 para reducir el efecto de ingresos extremadamente altos.

- **Variables de atrasos (30-59, 60-89, 90+ días)**: se identificó que exactamente 264 registros tienen el valor **98** simultáneamente en las tres variables. Este es un código especial que no representa 98 atrasos reales. Dado que estos registros tienen una tasa de incumplimiento del 54% (frente al 6.7% del resto del dataset), **eliminarlos o reemplazarlos con cero destruiría la señal más fuerte del dataset**. Se crea una variable indicadora (`flag_atrasos_98`) para preservar esa información, y luego se capan los valores al máximo real observado (13).

In [ ]:
########################################################
## TRATAMIENTO DE OUTLIERS                           ##
########################################################

# -- Variables de atrasos: tratamiento del código especial 98 --
# Los 264 registros con valor 98 en las 3 variables simultáneamente
# tienen una tasa de incumplimiento del 54%. Se preserva esa señal
# con una variable indicadora antes de hacer cualquier recorte.

cols_atrasos = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

mascara_98 = (
    (train['NumberOfTime30-59DaysPastDueNotWorse'] == 98) &
    (train['NumberOfTime60-89DaysPastDueNotWorse'] == 98) &
    (train['NumberOfTimes90DaysLate'] == 98)
)
train['flag_atrasos_98'] = mascara_98.astype(int)

mascara_98_test = (
    (test['NumberOfTime30-59DaysPastDueNotWorse'] == 98) &
    (test['NumberOfTime60-89DaysPastDueNotWorse'] == 98) &
    (test['NumberOfTimes90DaysLate'] == 98)
)
test['flag_atrasos_98'] = mascara_98_test.astype(int)

# Capar variables de atrasos al máximo real observado
for col in cols_atrasos:
    train[col] = train[col].clip(0, 13)
    test[col]  = test[col].clip(0, 13)

# -- RevolvingUtilizationOfUnsecuredLines --
# Se capa en 5 para conservar la señal de quienes superaron su límite
train['RevolvingUtilizationOfUnsecuredLines'] = train['RevolvingUtilizationOfUnsecuredLines'].clip(0, 5)
test['RevolvingUtilizationOfUnsecuredLines']  = test['RevolvingUtilizationOfUnsecuredLines'].clip(0, 5)

# -- DebtRatio --
cap_debtratio = train['DebtRatio'].quantile(0.99)
train['DebtRatio'] = train['DebtRatio'].clip(0, cap_debtratio)
test['DebtRatio']  = test['DebtRatio'].clip(0, cap_debtratio)

# -- age --
train['age'] = train['age'].clip(18, 95)
test['age']  = test['age'].clip(18, 95)

# -- MonthlyIncome --
cap_income = train['MonthlyIncome'].quantile(0.99)
train['MonthlyIncome'] = train['MonthlyIncome'].clip(0, cap_income)
test['MonthlyIncome']  = test['MonthlyIncome'].clip(0, cap_income)

print("Tratamiento de outliers aplicado")
print(f"  Registros con flag_atrasos_98 en train: {train['flag_atrasos_98'].sum()}")
print(f"  Registros con flag_ingreso_nulo en train: {train['flag_ingreso_nulo'].sum()}")
print(f"  Cap RevolvingUtilization: 5")
print(f"  Cap DebtRatio (P99):      {cap_debtratio:.2f}")
print(f"  Cap MonthlyIncome (P99):  {cap_income:.2f}")
print("\nEstadísticas después del tratamiento:")
train.describe().round(2)

### 3.3 Análisis de correlación y selección de características justificadas

Calculamos la correlación de Pearson entre todas las variables numéricas para entender qué tan relacionadas están entre sí y con la variable objetivo (**SeriousDlqin2yrs**).

In [ ]:
########################################################
## MAPA DE CORRELACIONES                             ##
########################################################

plt.figure(figsize=(13, 9))
correlaciones = train.corr()

sns.heatmap(correlaciones,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True)

plt.title('Mapa de Correlaciones', fontsize=14)
plt.tight_layout()
plt.show()

# Correlaciones con la variable objetivo ordenadas
print("\nCorrelación con SeriousDlqin2yrs:")
print(correlaciones['SeriousDlqin2yrs'].sort_values(ascending=False).round(3))

Del análisis de correlaciones se pueden extraer las siguientes conclusiones:

- Las **variables de atrasos** (30-59, 60-89, 90+ días) son las más correlacionadas con el incumplimiento. Sin embargo, también tienen una correlación alta entre sí (entre 0.58 y 0.75), lo que indica **multicolinealidad**. Para evitar redundancia, se excluye `NumberOfTime60-89DaysPastDueNotWorse` del modelo, conservando las otras dos.

- **RevolvingUtilizationOfUnsecuredLines** tiene la correlación individual más alta con el target, lo que confirma que el nivel de uso del crédito es un predictor muy relevante.

- **age** tiene correlación negativa: a mayor edad, menor riesgo de incumplimiento.

- **DebtRatio**, **NumberOfOpenCreditLinesAndLoans** y **NumberRealEstateLoansOrLines** presentan correlaciones bajas con el target, pero se conservan ya que aportan contexto financiero.

- Las variables indicadoras creadas en el preprocesamiento (`flag_ingreso_nulo`, `flag_atrasos_98`) se incluyen porque capturan información que de otro modo se perdería.

In [ ]:
########################################################
## SELECCIÓN DE VARIABLES                            ##
########################################################

variables = [
    'RevolvingUtilizationOfUnsecuredLines',  # uso del crédito — más predictiva
    'age',                                   # edad — correlación negativa
    'NumberOfTime30-59DaysPastDueNotWorse',  # historial de atrasos leves
    'NumberOfTimes90DaysLate',               # historial de atrasos graves
    'DebtRatio',                             # nivel de endeudamiento
    'MonthlyIncome',                         # capacidad de pago
    'NumberOfOpenCreditLinesAndLoans',       # exposición crediticia
    'NumberRealEstateLoansOrLines',          # compromiso hipotecario
    'NumberOfDependents',                    # carga familiar
    'flag_ingreso_nulo',                     # señal: ingreso era desconocido
    'flag_atrasos_98'                        # señal: código especial de riesgo alto
]

X = train[variables]
y = train['SeriousDlqin2yrs']

print('Variables seleccionadas para modelado:')
for v in variables:
    print(f'  - {v}')
print(f'\nDimensiones de X: {X.shape}')
print(f'Dimensiones de y: {y.shape}')

<a id="4-modelos"></a>
## 4. Implementación de Modelos

### 4.1 Preparación de datos para modelado

Separamos las variables predictoras (`X`) de la variable objetivo (`y`) y dividimos el conjunto de entrenamiento en dos partes:

- **X_train / y_train (80%):** datos con los que el modelo aprende.
- **X_val / y_val (20%):** datos que reservamos para evaluar qué tan bien generaliza el modelo antes de tocarlo con el conjunto de test.

Se usa `stratify=y` para garantizar que la proporción de defaults (6.7%) se mantenga igual en ambas partes del split, lo cual es fundamental cuando las clases están tan desbalanceadas.

In [ ]:
########################################################
## SPLIT TRAIN / VALIDACIÓN                          ##
########################################################

# Variables predictoras y objetivo
X = train[variables].copy()
y = train['SeriousDlqin2yrs'].copy()

# Conjunto de test (ya fue imputado y tratado en la Sección 3)
X_test = test[variables].copy()

# División interna: 80% entrenamiento, 20% validación
# stratify=y asegura la misma proporción de defaults en ambas partes
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

print('Dimensiones del split:')
print(f'  X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'\nProporción de defaults en y_train: {y_train.mean()*100:.2f}%')
print(f'Proporción de defaults en y_val:   {y_val.mean()*100:.2f}%')

### 4.2 k-Vecinos más cercanos (k-NN)

El algoritmo k-NN clasifica cada observación según las clases de sus `k` vecinos más cercanos. Como se basa en **distancias**, el escalamiento de las variables es fundamental: una variable con valores grandes dominaría el cálculo de distancia sobre las demás.

En esta sección evaluamos el impacto de dos tipos de escalamiento, tal como se vio en clase:

- **MinMaxScaler (Normalización):** transforma cada variable al rango [0, 1].
- **StandardScaler (Estandarización):** transforma cada variable para que tenga media 0 y desviación estándar 1.

En ambos casos el escalador se **ajusta solo con X_train** y se aplica a X_val, para no filtrar información del conjunto de validación al modelo.

In [ ]:
########################################################
## k-NN CON NORMALIZACIÓN (MinMaxScaler)             ##
########################################################

resultados_modelos = []

# Escalar
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_val_minmax   = scaler_minmax.transform(X_val)

# Entrenar modelo base con k=5
knn_minmax = KNeighborsClassifier(n_neighbors=5)
knn_minmax.fit(X_train_minmax, y_train)

# Predicciones
pred_knn_minmax = knn_minmax.predict(X_val_minmax)

# Resultados
print("===== k-NN con Normalización (MinMaxScaler) =====")
print(classification_report(y_val, pred_knn_minmax))
print("Matriz de confusión:")
print(confusion_matrix(y_val, pred_knn_minmax))

# Guardar F1 de clase 1 (default) para comparar modelos
from sklearn.metrics import f1_score
f1_knn_minmax = f1_score(y_val, pred_knn_minmax)
resultados_modelos.append({
    "Modelo":        "k-NN (MinMaxScaler)",
    "F1 (default)":  round(f1_knn_minmax, 4)
})

In [ ]:
########################################################
## k-NN CON ESTANDARIZACIÓN (StandardScaler)         ##
########################################################

# Escalar
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_val_std   = scaler_std.transform(X_val)

# Entrenar modelo base con k=5
knn_std = KNeighborsClassifier(n_neighbors=5)
knn_std.fit(X_train_std, y_train)

# Predicciones
pred_knn_std = knn_std.predict(X_val_std)

# Resultados
print("===== k-NN con Estandarización (StandardScaler) =====")
print(classification_report(y_val, pred_knn_std))
print("Matriz de confusión:")
print(confusion_matrix(y_val, pred_knn_std))

# Guardar F1 de clase 1 para comparar
f1_knn_std = f1_score(y_val, pred_knn_std)
resultados_modelos.append({
    "Modelo":        "k-NN (StandardScaler)",
    "F1 (default)":  round(f1_knn_std, 4)
})

### 4.3 Regresión Logística

La Regresión Logística estima la **probabilidad** de que una observación pertenezca a la clase 1 (default). A diferencia de k-NN, genera un modelo explícito con coeficientes que permiten interpretar el peso de cada variable.

También evaluamos los dos tipos de escalamiento para comparar su efecto, reutilizando los mismos `X_train_std`, `X_val_std`, `X_train_minmax` y `X_val_minmax` calculados en la sección anterior.

In [ ]:
########################################################
## REGRESIÓN LOGÍSTICA CON ESTANDARIZACIÓN           ##
########################################################

log_std = LogisticRegression(max_iter=50000, random_state=123)
log_std.fit(X_train_std, y_train)

pred_log_std = log_std.predict(X_val_std)

print("===== Regresión Logística con Estandarización =====")
print(classification_report(y_val, pred_log_std))
print("Matriz de confusión:")
print(confusion_matrix(y_val, pred_log_std))

f1_log_std = f1_score(y_val, pred_log_std)
resultados_modelos.append({
    "Modelo":        "Regresión Logística (StandardScaler)",
    "F1 (default)":  round(f1_log_std, 4)
})

In [ ]:
########################################################
## REGRESIÓN LOGÍSTICA CON NORMALIZACIÓN             ##
########################################################

log_minmax = LogisticRegression(max_iter=50000, random_state=123)
log_minmax.fit(X_train_minmax, y_train)

pred_log_minmax = log_minmax.predict(X_val_minmax)

print("===== Regresión Logística con Normalización =====")
print(classification_report(y_val, pred_log_minmax))
print("Matriz de confusión:")
print(confusion_matrix(y_val, pred_log_minmax))

f1_log_minmax = f1_score(y_val, pred_log_minmax)
resultados_modelos.append({
    "Modelo":        "Regresión Logística (MinMaxScaler)",
    "F1 (default)":  round(f1_log_minmax, 4)
})

### 4.4 Comparación inicial de modelos

Comparamos los cuatro modelos entrenados usando el **F1-Score de la clase 1 (default)** como métrica principal. Esta métrica es la más adecuada aquí porque combina precisión y recall, y no se ve distorsionada por el desbalanceo de clases como sí ocurre con el accuracy.

Esta comparación es solo un punto de partida — en la siguiente sección optimizaremos los hiperparámetros con Grid Search para mejorar estos resultados.

In [ ]:
########################################################
## TABLA COMPARATIVA — MODELOS BASE                  ##
########################################################

tabla_resultados = pd.DataFrame(resultados_modelos)
tabla_resultados = tabla_resultados.sort_values('F1 (default)', ascending=False)
tabla_resultados = tabla_resultados.reset_index(drop=True)

print('Comparación inicial de modelos (conjunto de validación):')
print(tabla_resultados.to_string(index=False))

<a id="5-optimización"></a>
## 5. Optimización de Hiperparámetros y Validación

En la sección anterior entrenamos modelos base con hiperparámetros fijos (`k=5`, `C=1`). En esta sección buscamos la **mejor combinación de hiperparámetros** para cada modelo usando dos herramientas vistas en clase:

- **StratifiedKFold (Validación Cruzada Estratificada):** divide los datos en 5 pliegues asegurando que cada uno mantenga la misma proporción de defaults. Así el resultado no depende de un único split sino del promedio de 5 evaluaciones diferentes.

- **GridSearchCV (Búsqueda en Grilla):** prueba todas las combinaciones posibles de hiperparámetros dentro de la grilla definida y se queda con la que produce el mejor resultado.

Usamos `scoring="f1"` como métrica de optimización porque, dado el fuerte desbalanceo de clases, el F1-Score de la clase 1 (default) es el indicador más honesto del desempeño real del modelo.

### 5.1 Grid Search para k-NN

Usamos un `Pipeline` que combina el escalador y el modelo en un solo objeto. Esto permite incluir el **tipo de escalamiento** (MinMax vs Standard) como un hiperparámetro más dentro de la grilla, y garantiza que el escalador se ajuste solo con los datos de entrenamiento en cada pliegue de la validación cruzada.

La grilla explora las siguientes combinaciones:

- **Escalador:** StandardScaler o MinMaxScaler
- **Número de vecinos k:** 5, 10, 15, 20
- **Pesos:** uniform (todos los vecinos pesan igual) o distance (vecinos más cercanos pesan más)
- **Métrica de distancia:** euclidiana o manhattan

In [ ]:
########################################################
## GRID SEARCH — k-NN                                ##
########################################################

# Estrategia de validación cruzada estratificada (5 pliegues)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# Pipeline: escalador + modelo en un solo objeto
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier())
])

# Grilla de hiperparámetros a explorar
param_grid_knn = {
    "scaler":           [StandardScaler(), MinMaxScaler()],
    "knn__n_neighbors": [5, 10, 15, 20],
    "knn__weights":     ["uniform", "distance"],
    "knn__metric":      ["euclidean", "manhattan"]
}

# Búsqueda en grilla con validación cruzada estratificada
grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid_knn,
    scoring="f1",       # optimizamos F1 de clase 1 (default)
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_train, y_train)

print("Mejores hiperparámetros k-NN:")
print(grid_knn.best_params_)
print(f"\nMejor F1 medio (CV): {grid_knn.best_score_:.4f}")

Revisamos las 10 mejores combinaciones de hiperparámetros encontradas para entender qué configuraciones funcionan mejor:

In [ ]:
########################################################
## TOP 10 — RESULTADOS GRID SEARCH k-NN              ##
########################################################

resultados_knn_cv = (
    pd.DataFrame(grid_knn.cv_results_)
    .sort_values("mean_test_score", ascending=False)
    [["params", "mean_test_score", "std_test_score"]]
    .rename(columns={
        "mean_test_score": "F1 medio (CV)",
        "std_test_score":  "Desviación"
    })
    .head(10)
    .reset_index(drop=True)
)
resultados_knn_cv

### 5.2 Grid Search para Regresión Logística

Para la Regresión Logística optimizamos los siguientes hiperparámetros:

- **C (regularización):** controla qué tan flexible es el modelo. Un C pequeño (0.01, 0.1) impone más restricciones y produce un modelo más simple. Un C grande (10, 100) da más libertad al modelo pero puede sobreajustarse.

- **class_weight:** permite indicarle al modelo que penalice más los errores en la clase minoritaria (default). Con `"balanced"` el modelo ajusta automáticamente los pesos según la proporción de cada clase — útil con datos desbalanceados.

In [ ]:
########################################################
## GRID SEARCH — REGRESIÓN LOGÍSTICA                 ##
########################################################

# Pipeline: escalador + modelo
pipe_log = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=50000, random_state=123))
])

# Grilla de hiperparámetros
param_grid_log = {
    "logreg__C":            [0.01, 0.1, 1.0, 10.0],
    "logreg__class_weight": [None, "balanced"]
}

# Búsqueda en grilla con validación cruzada estratificada
grid_log = GridSearchCV(
    estimator=pipe_log,
    param_grid=param_grid_log,
    scoring="f1",       # optimizamos F1 de clase 1 (default)
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_log.fit(X_train, y_train)

print("Mejores hiperparámetros Regresión Logística:")
print(grid_log.best_params_)
print(f"\nMejor F1 medio (CV): {grid_log.best_score_:.4f}")

Revisamos las 10 mejores combinaciones para la Regresión Logística:

In [ ]:
########################################################
## TOP 10 — RESULTADOS GRID SEARCH REGRESIÓN LOG.    ##
########################################################

resultados_log_cv = (
    pd.DataFrame(grid_log.cv_results_)
    .sort_values("mean_test_score", ascending=False)
    [["params", "mean_test_score", "std_test_score"]]
    .rename(columns={
        "mean_test_score": "F1 medio (CV)",
        "std_test_score":  "Desviación"
    })
    .head(10)
    .reset_index(drop=True)
)
resultados_log_cv

### 5.3 Comparación de modelos tras validación cruzada

Comparamos los **mejores modelos** de cada algoritmo según el F1 medio obtenido en la validación cruzada. El modelo con mayor F1 será el que usemos en la Sección 6 para la evaluación final.

In [ ]:
########################################################
## COMPARACIÓN FINAL TRAS GRID SEARCH                ##
########################################################

resumen_cv = pd.DataFrame([
    {
        "Modelo":         "k-NN (mejor configuración CV)",
        "Mejores params": str(grid_knn.best_params_),
        "F1 medio (CV)":  round(grid_knn.best_score_, 4)
    },
    {
        "Modelo":         "Regresión Logística (mejor configuración CV)",
        "Mejores params": str(grid_log.best_params_),
        "F1 medio (CV)":  round(grid_log.best_score_, 4)
    }
])

print("Comparación de modelos — F1 medio en validación cruzada:")
resumen_cv

<a id="6-métricas"></a>
## 6. Métricas de Evaluación y Decisión

En esta sección tomamos el **mejor modelo encontrado en la Sección 5** y lo evaluamos sobre el conjunto de validación `X_val` que fue reservado exclusivamente para esta etapa — el modelo nunca lo vio durante el entrenamiento.

Para evaluar el desempeño usamos:

- **Matriz de confusión:** muestra cuántos casos el modelo clasificó correctamente e incorrectamente para cada clase.
- **Reporte de clasificación:** resume las métricas de Accuracy, Precisión, Recall y F1-Score para cada clase.

### 6.1 Evaluación del modelo final

Comparamos los dos mejores modelos obtenidos por Grid Search — k-NN y Regresión Logística — evaluándolos sobre el conjunto de validación.

Recordemos el significado de cada métrica en el contexto de riesgo crediticio:

- **Accuracy:** porcentaje total de predicciones correctas. Con datos desbalanceados puede ser engañosa — un modelo que nunca predice default tendría 93% de accuracy sin detectar ningún caso de riesgo.
- **Precisión (clase 1):** de todos los que el modelo marcó como "default", ¿qué proporción realmente incumplió?
- **Recall (clase 1):** de todos los que realmente incumplieron, ¿qué proporción detectó el modelo? Esta métrica es crítica en riesgo crediticio porque un Falso Negativo (no detectar a alguien que va a incumplir) es más costoso que un Falso Positivo.
- **F1-Score (clase 1):** promedio armónico entre precisión y recall. Es la métrica principal para comparar modelos con clases desbalanceadas.

In [ ]:
########################################################
## EVALUACIÓN — MEJOR MODELO k-NN                   ##
########################################################

# Recuperar el mejor modelo k-NN y evaluarlo en validación
mejor_knn = grid_knn.best_estimator_
mejor_knn.fit(X_train, y_train)
pred_val_knn = mejor_knn.predict(X_val)

print("===== Mejor k-NN (Grid Search) — Conjunto de Validación =====")
print(classification_report(y_val, pred_val_knn))

In [ ]:
########################################################
## EVALUACIÓN — MEJOR MODELO REGRESIÓN LOGÍSTICA    ##
########################################################

# Recuperar el mejor modelo de Regresión Logística y evaluarlo en validación
mejor_log = grid_log.best_estimator_
mejor_log.fit(X_train, y_train)
pred_val_log = mejor_log.predict(X_val)

print("===== Mejor Regresión Logística (Grid Search) — Conjunto de Validación =====")
print(classification_report(y_val, pred_val_log))

### 6.2 Matrices de confusión

La matriz de confusión muestra en detalle los cuatro tipos de resultado posibles:

- **Verdadero Negativo (TN):** predijo "no default" y era correcto.
- **Falso Positivo (FP):** predijo "default" pero no lo era. *(El banco rechaza a un cliente que habría pagado)*
- **Falso Negativo (FN):** predijo "no default" pero sí incumplió. *(El banco aprueba a alguien que no va a pagar — el error más costoso)*
- **Verdadero Positivo (TP):** predijo "default" y era correcto.

In [ ]:
########################################################
## MATRICES DE CONFUSIÓN — AMBOS MODELOS            ##
########################################################

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

modelos_eval = [
    (pred_val_knn, "Mejor k-NN (Grid Search)"),
    (pred_val_log, "Mejor Regresión Logística (Grid Search)")
]

for ax, (pred, titulo) in zip(axes, modelos_eval):
    mc = confusion_matrix(y_val, pred)
    sns.heatmap(
        mc,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        annot_kws={"size": 14},
        xticklabels=["No default (0)", "Default (1)"],
        yticklabels=["No default (0)", "Default (1)"]
    )
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Valor real", fontsize=10)

plt.suptitle("Matrices de Confusión — Conjunto de Validación",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.3 Decisión del modelo final

Con base en las métricas de validación cruzada (Sección 5) y la evaluación sobre el conjunto de validación (Sección 6), seleccionamos el modelo con mejor F1-Score en la clase 1 como **modelo final**.

En el contexto de riesgo crediticio priorizamos el **Recall de la clase 1**: es preferible ser más cauteloso y señalar algunos falsos positivos que dejar pasar clientes que van a incumplir (falsos negativos).

In [ ]:
########################################################
## TABLA COMPARATIVA FINAL                          ##
########################################################

from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score

comparacion_final = pd.DataFrame([
    {
        "Modelo":              "k-NN (mejor Grid Search)",
        "Accuracy":            round(accuracy_score(y_val, pred_val_knn), 4),
        "Precisión (default)": round(precision_score(y_val, pred_val_knn), 4),
        "Recall (default)":    round(recall_score(y_val, pred_val_knn), 4),
        "F1 (default)":        round(f1_score(y_val, pred_val_knn), 4)
    },
    {
        "Modelo":              "Regresión Logística (mejor Grid Search)",
        "Accuracy":            round(accuracy_score(y_val, pred_val_log), 4),
        "Precisión (default)": round(precision_score(y_val, pred_val_log), 4),
        "Recall (default)":    round(recall_score(y_val, pred_val_log), 4),
        "F1 (default)":        round(f1_score(y_val, pred_val_log), 4)
    }
])

print("Comparación final de modelos — Conjunto de Validación:")
comparacion_final

<a id="7-test"></a>
## 7. Predicción en el conjunto de test

Una vez seleccionado el modelo final, lo entrenamos con **todo el conjunto de entrenamiento** (no solo el 80%) para aprovechar al máximo los datos disponibles antes de generar las predicciones sobre el conjunto de test.

El resultado es un archivo `submission.csv` con el formato requerido: una columna `Id` y una columna `Probability` con la probabilidad estimada de incumplimiento para cada persona del conjunto de test.

In [ ]:
########################################################
## ENTRENAMIENTO FINAL CON TODOS LOS DATOS           ##
########################################################

# Seleccionar el mejor modelo según los resultados de la Sección 6
# (reemplazar grid_log por grid_knn si k-NN obtuvo mejor F1)
modelo_final = grid_log.best_estimator_

# Entrenar con todo X (no solo X_train) para maximizar la información
modelo_final.fit(X, y)

# Generar probabilidades de default para el conjunto de test
proba_test = modelo_final.predict_proba(X_test)[:, 1]

print(f"Predicciones generadas: {len(proba_test)}")
print(f"Probabilidad mínima:    {proba_test.min():.4f}")
print(f"Probabilidad máxima:    {proba_test.max():.4f}")
print(f"Probabilidad media:     {proba_test.mean():.4f}")

In [ ]:
########################################################
## CONSTRUCCIÓN DEL ARCHIVO SUBMISSION               ##
########################################################

submission = pd.DataFrame({
    "Id":          range(1, len(proba_test) + 1),
    "Probability": proba_test
})

print("Primeras filas del archivo de submission:")
submission.head(10)

In [ ]:
########################################################
## DESCARGA DEL ARCHIVO DESDE GOOGLE COLAB           ##
########################################################

# Guardar el archivo en el entorno de Colab
submission.to_csv("submission.csv", index=False)

# Descargar directamente al computador
from google.colab import files
files.download("submission.csv")

print("Archivo submission.csv generado y descargado.")